# Phase 6c — SQuAD 2.0 Controlled Stress Benchmark (Chunk 2)

This notebook calibrates retrieval-failure heuristics on a fixed **SQuAD 2.0 train-only** development split, freezes the selected detector, and then compares it with the original Phase 5 detector on held-out validation stress tracks. It uses only free local models and performs no answer generation or answer-quality evaluation.

## 1. Colab environment

Select **Runtime → Change runtime type → T4 GPU**, set `REPOSITORY_URL`, and run all cells.

In [ ]:
from pathlib import Path
import csv
import json
import os
import random
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/YOUR_GITHUB_USERNAME/adaptive-rag.git'
PROJECT_DIR = Path('/content/adaptive-rag')

if not (PROJECT_DIR / 'src').exists():
    if 'YOUR_GITHUB_USERNAME' in REPOSITORY_URL:
        raise RuntimeError('Set REPOSITORY_URL, then run this cell again.')
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Working directory:', Path.cwd())

In [ ]:
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'unavailable')

## 2. Fixed experiment configuration

Full mode uses 5,000 passages and screens 5,000 TRAIN questions to target 600 healthy, 150 objective retrieval failures, and 200 controlled missing-evidence examples. It refuses to calibrate with fewer than 100 genuine retrieval failures. A fixed pool of 2,500 validation hard-distractor candidates is screened to select up to 100 objectively induced failures; all other validation tracks remain fixed. `SMOKE_TEST=True` uses smaller pools with nonzero minimum-support guards. Saved result files are reused unless `FORCE_RERUN=True`.

In [ ]:
SMOKE_TEST = False
FORCE_RERUN = False
RESULTS_DIR = Path('results')
OUTPUT_PATHS = {
    'manifest': RESULTS_DIR / 'squad_stress_manifest.json',
    'calibration_diagnostics': RESULTS_DIR / 'squad_calibration_diagnostics.csv',
    'calibration': RESULTS_DIR / 'squad_detector_calibration.json',
    'per_example': RESULTS_DIR / 'squad_stress_per_example.csv',
    'metrics': RESULTS_DIR / 'squad_stress_metrics.json',
    'confusion': RESULTS_DIR / 'squad_failure_confusion.csv',
}
REUSE_SAVED = (
    not FORCE_RERUN
    and OUTPUT_PATHS['per_example'].exists()
    and OUTPUT_PATHS['metrics'].exists()
)
print({'smoke_test': SMOKE_TEST, 'reuse_saved_results': REUSE_SAVED})

## 3. Prepare disjoint train calibration and validation stress samples

Stable context hashes deduplicate passages. The manifest saves every sampled question ID, corpus document ID, paraphrase, and selected hard-distractor ID. Calibration never reads validation outcomes.

In [ ]:
from dataclasses import asdict
from src.evaluation import (
    SquadStressSamplingConfig,
    StressTrack,
    prepare_squad_stress_dataset,
)

sampling = SquadStressSamplingConfig(
    corpus_size=500 if SMOKE_TEST else 5_000,
    calibration_question_count=60 if SMOKE_TEST else 600,
    calibration_screening_count=600 if SMOKE_TEST else 5_000,
    calibration_failure_target_count=10 if SMOKE_TEST else 150,
    calibration_minimum_failure_count=5 if SMOKE_TEST else 100,
    calibration_missing_count=20 if SMOKE_TEST else 200,
    calibration_paraphrase_attempt_count=80 if SMOKE_TEST else 500,
    hard_distractor_count=12 if SMOKE_TEST else 100,
    hard_distractor_candidate_count=300 if SMOKE_TEST else 2_500,
    hard_distractor_minimum_induced_count=1 if SMOKE_TEST else 25,
    paraphrase_pair_count=6 if SMOKE_TEST else 50,
    controlled_missing_count=12 if SMOKE_TEST else 100,
    natural_unanswerable_count=12 if SMOKE_TEST else 100,
    hard_distractors_per_query=10 if SMOKE_TEST else 20,
    seed=SEED,
    cache_dir='.cache/huggingface',
)
desired_sampling_config = asdict(sampling)
desired_sampling_config['cache_dir'] = str(desired_sampling_config['cache_dir'])
if REUSE_SAVED:
    saved_preview = json.loads(OUTPUT_PATHS['metrics'].read_text(encoding='utf-8'))
    saved_sampling_config = saved_preview.get('train_development_calibration', {}).get('sampling_config')
    REUSE_SAVED = saved_sampling_config == desired_sampling_config
    if not REUSE_SAVED:
        print('Saved results use a different sampling config; running the requested config.')
stress_data = None
if REUSE_SAVED:
    print('Reusing saved metrics and per-example rows; dataset/model work is skipped.')
else:
    stress_data = prepare_squad_stress_dataset(sampling, OUTPUT_PATHS['manifest'])
    train_ids = {item.id for item in stress_data.calibration_questions}
    validation_ids = {item.id for item in stress_data.validation_questions}
    assert train_ids.isdisjoint(validation_ids), 'Train/validation question IDs overlap.'
    print('TRAIN screening questions:', len(train_ids))
    print('TRAIN controlled-missing target:', sampling.calibration_missing_count)
    print('Validation candidate observations:', len(stress_data.validation_questions))
    print('Corpus passages:', len(stress_data.documents))
    print('Manifest:', stress_data.manifest_path)

## 4. Initialize free local retrieval and rewrite components

The dense index, embedding model, cross-encoder, and local Qwen model are reused. The exclusion wrapper removes the stable gold-context ID from both dense and BM25 results for controlled missing-evidence cases, preventing duplicate-ID leakage.

In [ ]:
from src.evaluation import (
    CachedCrossEncoderReranker,
    CachedDenseRetriever,
    CachedQueryRewriter,
    ExcludingRetriever,
    LocalQwenParaphraser,
    RetrievalExclusionController,
)
from src.rag import (
    BM25Retriever,
    CrossEncoderReranker,
    FAISSRetriever,
    HybridRetriever,
    LocalQwenGenerator,
    LocalQwenQueryRewriter,
    RAGConfig,
)

cached_dense = filtered_dense = filtered_bm25 = hybrid = reranker = None
exclusion_controller = generator = cached_rewriter = None
if not REUSE_SAVED:
    rag_config = RAGConfig(
        embedding_model_name='sentence-transformers/all-MiniLM-L6-v2',
        generation_model_name='Qwen/Qwen2.5-1.5B-Instruct',
        embedding_batch_size=128,
        max_new_tokens=64,
        do_sample=False,
    )
    index_dir = Path('.cache/squad_stress_faiss') / stress_data.fingerprint
    base_dense = FAISSRetriever(
        rag_config.embedding_model_name, device=rag_config.device, batch_size=128
    )
    if (index_dir / 'documents.faiss').exists():
        base_dense.load(index_dir)
    else:
        base_dense.build(stress_data.documents)
        base_dense.save(index_dir)
    cached_dense = CachedDenseRetriever(base_dense)
    base_bm25 = BM25Retriever(stress_data.documents)
    filtered_dense = ExcludingRetriever(cached_dense)
    filtered_bm25 = ExcludingRetriever(base_bm25)
    exclusion_controller = RetrievalExclusionController(filtered_dense, filtered_bm25)
    hybrid = HybridRetriever(filtered_dense, filtered_bm25)
    reranker = CachedCrossEncoderReranker(CrossEncoderReranker(batch_size=64))
    generator = LocalQwenGenerator(rag_config)
    cached_rewriter = CachedQueryRewriter(LocalQwenQueryRewriter(generator))
    print('FAISS index:', index_dir)
    print('Cross-encoder device:', reranker.reranker.device)
    print('Local Qwen device:', generator.device)

## 5. Build reproducible hard-distractor and paraphrase tracks

Hard distractors prefer same-title passages, then embedding-nearest non-gold passages. Paraphrases are generated deterministically by local Qwen without answer text in the prompt. Both are persisted in the manifest.

In [ ]:
from src.evaluation import prepare_paraphrases, select_hard_distractors

paraphrases = {}
hard_distractors = {}
hard_selection = None
if not REUSE_SAVED:
    paraphrases = prepare_paraphrases(
        stress_data,
        LocalQwenParaphraser(generator),
        progress_callback=lambda i, n, q: print(f'Paraphrase {i}/{n}') if i % 10 == 0 or i == n else None,
    )
    hard_selection = select_hard_distractors(
        stress_data,
        cached_dense,
        filtered_dense,
        filtered_bm25,
        hybrid,
        reranker,
        exclusion_controller,
        per_question=sampling.hard_distractors_per_query,
        search_depth=100,
        target_count=sampling.hard_distractor_count,
        minimum_induced_count=sampling.hard_distractor_minimum_induced_count,
        cutoff=5,
        progress_callback=lambda i, n, q: print(f'Hard screen {i}/{n}') if i % 25 == 0 or i == n else None,
    )
    hard_distractors = hard_selection.selected_distractors
    selected_validation_questions = [
        item for item in stress_data.validation_questions
        if item.track != StressTrack.HARD_DISTRACTOR or item.id in hard_distractors
    ]
    active_validation_queries = [
        paraphrases.get(item.id, item.original_query)
        if item.track == StressTrack.PARAPHRASE_TRANSFORMED
        else item.original_query
        for item in selected_validation_questions
    ]
    all_queries = [item.original_query for item in stress_data.calibration_questions]
    all_queries.extend(active_validation_queries)
    cached_dense.preload(all_queries, top_k=40)
    print('Saved paraphrases:', len(paraphrases))
    print('Hard-distractor candidate cases:', hard_selection.candidate_count)
    print('Hard-distractor induced failures in candidate pool:', hard_selection.induced_failure_count)
    print('Hard-distractor selected induced cases:', hard_selection.selected_count)

## 6. TRAIN-only diagnostic export and calibration

The objective label comes from gold rank at cutoff 5, or from controlled removal. Candidate thresholds are train-derived diagnostic quantiles. A fixed-seed lightweight search prioritizes failure recall, healthy specificity, correct insufficient-evidence detection, and objective-mode macro-F1. The original Phase 5 config remains available. No validation diagnostics are consulted before the selected config is saved and frozen.

In [ ]:
from src.evaluation import (
    calibrate_failure_detector,
    collect_calibration_diagnostics,
    save_detector_calibration,
)
from src.rag import RetrievalFailureDetector

calibration_result = None
calibration_dataset = None
calibration_examples = []
if not REUSE_SAVED:
    original_detector = RetrievalFailureDetector()
    calibration_dataset = collect_calibration_diagnostics(
        stress_data,
        filtered_dense,
        filtered_bm25,
        hybrid,
        reranker,
        exclusion_controller,
        original_detector,
        paraphraser=LocalQwenParaphraser(generator),
        healthy_target_count=sampling.calibration_question_count,
        failure_target_count=sampling.calibration_failure_target_count,
        minimum_failure_count=sampling.calibration_minimum_failure_count,
        missing_target_count=sampling.calibration_missing_count,
        paraphrase_attempt_count=sampling.calibration_paraphrase_attempt_count,
        seed=SEED,
        output_path=OUTPUT_PATHS['calibration_diagnostics'],
        cutoff=5,
        progress_callback=lambda i, n, q: print(f'Calibration {i}/{n}') if i % 100 == 0 or i == n else None,
    )
    calibration_examples = list(calibration_dataset.examples)
    calibration_rows = list(calibration_dataset.rows)
    calibration_result = calibrate_failure_detector(
        calibration_examples,
        original_config=original_detector.config,
        seed=SEED,
        candidate_count=256 if SMOKE_TEST else 1_024,
        maximum_healthy_false_positive_rate=0.15,
    )
    save_detector_calibration(calibration_result, OUTPUT_PATHS['calibration'])
    print('TRAIN screening/construction:', json.dumps(calibration_dataset.construction_summary, indent=2))
    print('TRAIN calibration class support:', calibration_dataset.class_support)
    def print_calibration_metrics(label, metrics):
        print(label, {
            'class_support': metrics.class_support,
            'binary_precision': metrics.binary.precision,
            'binary_recall': metrics.binary.recall,
            'binary_f1': metrics.binary.f1,
            'healthy_false_positive_rate': metrics.binary.false_positive_rate,
            'retrieval_failure_recall': metrics.retrieval_failure_recall,
            'insufficient_evidence_recall': metrics.insufficient_evidence_recall,
            'failure_mode_macro_f1': metrics.failure_mode.macro_f1,
        })
    print_calibration_metrics('ORIGINAL_PHASE5 TRAIN', calibration_result.original_train_metrics)
    print_calibration_metrics('TRAIN_CALIBRATED TRAIN', calibration_result.selected_train_metrics)
    print('Frozen TRAIN-CALIBRATED config:', json.dumps(calibration_result.to_dict()['selected_config'], indent=2))
    for label, signals in calibration_result.diagnostic_distributions.items():
        print(label, {name: {'count': stats['count'], 'p50': stats['p50']} for name, stats in signals.items()})

## 7. Freeze both detectors and run held-out validation stress tracks

Both workflows use the same cached retrieval, reranking, and deterministic rewrite components. They differ only in failure-detector configuration. Recovery is retrieval-only and bounded to one retry.

In [ ]:
from src.evaluation import SquadStressBenchmark
from src.rag import SelfHealingRAGWorkflow, SelfHealingWorkflowConfig

records = []
payload = None
if REUSE_SAVED:
    payload = json.loads(OUTPUT_PATHS['metrics'].read_text(encoding='utf-8'))
    with OUTPUT_PATHS['per_example'].open(encoding='utf-8', newline='') as handle:
        result_rows = list(csv.DictReader(handle))
else:
    calibrated_detector = RetrievalFailureDetector(calibration_result.selected_config)
    workflow_config = SelfHealingWorkflowConfig(
        initial_retrieval_depth=20,
        expanded_retrieval_depth=40,
        reranked_top_k=10,
        generation_top_k=10,
        max_retries=1,
    )
    original_workflow = SelfHealingRAGWorkflow(
        filtered_dense, filtered_bm25, hybrid, reranker, generator,
        failure_detector=original_detector,
        config=workflow_config,
        query_rewriter=cached_rewriter,
    )
    calibrated_workflow = SelfHealingRAGWorkflow(
        filtered_dense, filtered_bm25, hybrid, reranker, generator,
        failure_detector=calibrated_detector,
        config=workflow_config,
        query_rewriter=cached_rewriter,
    )
    runner = SquadStressBenchmark(
        filtered_dense, filtered_bm25, hybrid, reranker, exclusion_controller,
        original_detector, calibrated_detector, original_workflow, calibrated_workflow,
        cutoff=5, output_dir=RESULTS_DIR,
    )
    calibration_summary = {
        'split': 'SQuAD 2.0 TRAIN only',
        'sampling_config': desired_sampling_config,
        'screened_question_count': len(stress_data.calibration_questions),
        'selected_class_support': calibration_dataset.class_support,
        'construction': calibration_dataset.construction_summary,
        'diagnostic_observation_count': len(calibration_examples),
        'detector_calibration': calibration_result.to_dict(),
    }
    records, payload = runner.run(
        stress_data, hard_distractors, paraphrases, calibration_summary,
        stress_construction_summary={
            'hard_distractor_candidate_count': hard_selection.candidate_count,
            'hard_distractor_induced_failure_count': hard_selection.induced_failure_count,
            'hard_distractor_selected_count': hard_selection.selected_count,
            'paraphrase_pair_count': sampling.paraphrase_pair_count,
        },
        progress_callback=lambda i, n, q: print(f'Validation {i}/{n}: {q.track.value}') if i % 25 == 0 or i == n else None,
    )
    from dataclasses import asdict
    result_rows = [asdict(item) for item in records]
print('Loaded/running rows:', len(result_rows))

## 8. Required train/validation summary

Natural unanswerable behavior is descriptive only: unanswerable relative to its source paragraph does not prove absence from the entire corpus. Its binary and objective-mode scores are therefore intentionally unavailable.

In [ ]:
train_summary = payload['train_development_calibration']
print('=== TRAIN DEVELOPMENT / CALIBRATION (not final evaluation) ===')
print(json.dumps(train_summary, indent=2))

unique_track_examples = {}
for row in result_rows:
    key = (str(row['track']), str(row['example_id']))
    unique_track_examples[key] = True
track_counts = {}
for track, _ in unique_track_examples:
    track_counts[track] = track_counts.get(track, 0) + 1
print('\n=== HELD-OUT VALIDATION STRESS COUNTS ===')
print(json.dumps(track_counts, indent=2))
print('Stress construction:', json.dumps(payload.get('stress_construction', {}), indent=2))
reference_variant = next(iter(payload['held_out_validation_stress']))
paraphrase_metrics = payload['held_out_validation_stress'][reference_variant].get('PARAPHRASE_TRANSFORMED', {})
print('Paraphrase-induced failures@5:', paraphrase_metrics.get('paraphrase_induced_failures_at_5', 0))

print('\n=== HELD-OUT VALIDATION METRICS ===')
for variant, tracks in payload['held_out_validation_stress'].items():
    print(f'\n{variant}')
    for track, metrics in tracks.items():
        binary = metrics['binary_failure_detection']
        mode = metrics['failure_mode']
        print(track, {
            'examples': metrics['example_count'],
            'initial_failure_rate': metrics['initial_failure_rate'],
            'precision': binary['precision'] if binary else None,
            'recall': binary['recall'] if binary else None,
            'f1': binary['f1'] if binary else None,
            'mode_macro_f1': mode['macro_f1'] if mode else None,
            'recovery_attempts': metrics['recovery_attempts'],
            'recovered_at_5': metrics['recovered_at_5'],
            'failed_recovery_at_5': metrics['failed_recovery_at_5'],
            'rank_movement': metrics['gold_rank_movement'],
            'false_positive_healing_rate': metrics['false_positive_healing_rate'],
            'paraphrase_rank_movement': metrics['paraphrase_gold_rank_movement'],
        })

## 9. Representative cases for inspection

In [ ]:
def is_true(value):
    return value is True or str(value).lower() == 'true'

def retry_count(row):
    return int(row.get('retry_count') or 0)

def show_cases(label, predicate, limit=3):
    print(f'\n{label}')
    selected = [row for row in result_rows if predicate(row)][:limit]
    if not selected:
        print('  none observed')
        return
    for row in selected:
        print({key: row.get(key) for key in (
            'example_id', 'track', 'detector_variant', 'active_query',
            'objective_failure_label', 'predicted_failure_type',
            'initial_gold_rank', 'final_gold_rank', 'retry_count', 'graph_path'
        )})

objective_failures = {'RETRIEVAL_FAILURE', 'MISSING_EVIDENCE'}
show_cases('Correctly detected failures', lambda r: r['objective_failure_label'] in objective_failures and r['predicted_failure_type'] != 'HEALTHY')
show_cases('Missed failures', lambda r: r['objective_failure_label'] in objective_failures and r['predicted_failure_type'] == 'HEALTHY')
show_cases('Successful recovery@5', lambda r: is_true(r['recovered_at_5']))
show_cases('Failed recovery@5', lambda r: is_true(r['failed_recovery_at_5']))
show_cases('False-positive healing', lambda r: r['objective_failure_label'] == 'HEALTHY' and retry_count(r) > 0)

## 10. Outputs and runtime

The first full T4 run is expected to take roughly **60–150 minutes**, depending mainly on how many TRAIN paraphrases are needed to reach objective failure support. Smoke mode is typically about **15–30 minutes**. Later runs reuse Hugging Face, FAISS, paraphrase, rewrite, reranker, and result caches where applicable. No model weights or large caches are committed.

In [ ]:
print('Artifacts:')
for name, path in OUTPUT_PATHS.items():
    print(f'- {name}: {path} (exists={path.exists()})')
print('\nChunk 2 complete: retrieval diagnostics, calibration, stress detection, and retrieval-only recovery only.')